In [1]:
# Verificar GPU disponible
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("ADVERTENCIA: No se detectó GPU. El entrenamiento será muy lento.")

CUDA disponible: True
GPU: NVIDIA GeForce RTX 3060
VRAM: 12.9 GB


In [4]:
import torch
from transformers import BertForMaskedLM, BertTokenizer, AutoTokenizer, DataCollatorWithPadding, TrainingArguments, Trainer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import Dataset
import numpy as np
import evaluate

In [ ]:
# Solo ejecutar si no están instaladas las dependencias
%pip install evaluate 
# pip install evaluate transformers datasets torch pandas scikit-learn --quiet

#### Load Preprocessing saved (dataset['text'] tokenizados y labels mapeadas con ID)

In [5]:
from datasets import load_from_disk
import json

# Cargar datasets tokenizados
train_dataset = load_from_disk("data_processed/train_tokenized")
val_dataset = load_from_disk("data_processed/val_tokenized")
test_dataset = load_from_disk("data_processed/test_tokenized")

# Cargar mappings de labels
with open("data_processed/label_to_id.json", "r") as f:
    label_to_id = json.load(f)
with open("data_processed/id_to_label.json", "r") as f:
    id_to_label = json.load(f)
    # Convertir keys a int (JSON las guarda como string)
    id_to_label = {int(k): v for k, v in id_to_label.items()}

print(f"Train: {len(train_dataset):,}")
print(f"Val: {len(val_dataset):,}")
print(f"Test: {len(test_dataset):,}")
print(f"Labels: {len(label_to_id)}")

Train: 8,999,985
Val: 499,999
Test: 500,000
Labels: 1571


#### Import and split

- Uses Pandas DataFrames to split the data into train, validation and test using train_test_split() from sklearn and its stratify parameter for preserving class proportions.

- Converts DataFrames to Datasets and renames columns to aling data structure to be able to use Hugging Face's pre-train model.

In [5]:
# Cargar dataset (ruta local)
df_input = pd.read_csv('train.csv')
print(f"Dataset total: {len(df_input):,} registros")

# Filtrar solo español
df_input = df_input[df_input['language'] == 'spanish']
print(f"Solo español: {len(df_input):,} registros")

# DATASET COMPLETO SIN FILTROS DE CATEGORÍAS
df_input_full = df_input.drop(["language", "label_quality"], axis=1)

# Filtro técnico: mínimo 20 ejemplos para soportar los 2 splits estratificados
MIN_SAMPLES = 20

category_counts = df_input_full['category'].value_counts()
categories_removed = category_counts[category_counts < MIN_SAMPLES]

print(f"\nCategorías eliminadas (menos de {MIN_SAMPLES} ejemplos): {len(categories_removed)}")
if len(categories_removed) > 0:
    print("Detalle:")
    for cat, count in categories_removed.items():
        print(f"  - {cat}: {count} ejemplos")

valid_categories = category_counts[category_counts >= MIN_SAMPLES].index
df_input_full = df_input_full[df_input_full['category'].isin(valid_categories)]

print(f"\nRegistros después de filtro: {len(df_input_full):,}")
print(f"Total categorías: {df_input_full['category'].nunique()}")

# Split estratificado: 90% train, 5% val, 5% test
train_df, temp_df = train_test_split(
    df_input_full, 
    test_size=0.1,  # 10% para val+test
    random_state=42, 
    stratify=df_input_full['category']
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5,  # 50% de temp = 5% del total
    random_state=42, 
    stratify=temp_df['category']
)

# Convertir a Datasets de HuggingFace
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# Renombrar columnas
train_dataset = train_dataset.rename_column("title", "text").rename_column("category", "labels")
val_dataset = val_dataset.rename_column("title", "text").rename_column("category", "labels")
test_dataset = test_dataset.rename_column("title", "text").rename_column("category", "labels")

print(f"\nSplit final:")
print(f"  Train: {len(train_dataset):,}")
print(f"  Val:   {len(val_dataset):,}")
print(f"  Test:  {len(test_dataset):,}")

Dataset total: 20,000,000 registros
Solo español: 10,000,000 registros

Categorías eliminadas (menos de 20 ejemplos): 3
Detalle:
  - SNACK_HOLDERS: 9 ejemplos
  - ANTI_STATIC_PLIERS: 5 ejemplos
  - CARD_PAYMENT_TERMINALS: 2 ejemplos

Registros después de filtro: 9,999,984
Total categorías: 1571

Split final:
  Train: 8,999,985
  Val:   499,999
  Test:  500,000


#### Preprocess

- Load a BETO cased tokenizer to preprocess the text field.
- Tokenize, pad, and truncate for training.
- Create a map of the expected ids to their labels.

In [6]:
# Create a mapping from category names to ids
label_to_id = {label: id for id, label in enumerate(train_dataset.unique('labels'))}
id_to_label = {id: label for label, id in label_to_id.items()}
print(f"Total labels: {len(label_to_id)}")

Total labels: 1571


In [7]:
tokenizer = AutoTokenizer.from_pretrained('dccuchile/bert-base-spanish-wwm-cased')
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    'dccuchile/bert-base-spanish-wwm-cased',
    num_labels = len(train_dataset.unique('labels')),
    id2label = id_to_label,
    label2id = label_to_id)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # convert the logits to their predicted class
    predictions = np.argmax(logits, axis=-1) # TODO: axis = -1 ?
    # Use the mapped labels for computation
    return accuracy.compute(predictions=predictions, references=labels)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# Map string labels to integers
def map_labels_to_ids(examples):
    return {'labels': label_to_id[examples['labels']]}

print("Tokenizando train...")
train_dataset = train_dataset.map(preprocess_function, batched=True, batch_size=1000)
print("Tokenizando val...")
val_dataset = val_dataset.map(preprocess_function, batched=True, batch_size=1000)
print("Tokenizando test...")
test_dataset = test_dataset.map(preprocess_function, batched=True, batch_size=1000)

print("Mapeando labels...")
train_dataset = train_dataset.map(map_labels_to_ids)
val_dataset = val_dataset.map(map_labels_to_ids)
test_dataset = test_dataset.map(map_labels_to_ids)

print("Preprocesamiento completo!")

Tokenizando train...


Map: 100%|██████████| 8999985/8999985 [04:05<00:00, 36718.07 examples/s]


Tokenizando val...


Map: 100%|██████████| 499999/499999 [00:13<00:00, 37310.90 examples/s]


Tokenizando test...


Map: 100%|██████████| 500000/500000 [00:13<00:00, 36764.56 examples/s]


Mapeando labels...


Map: 100%|██████████| 500000/500000 [00:27<00:00, 18152.53 examples/s]

Preprocesamiento completo!


In [15]:
# Guardar datasets tokenizados a disco
train_dataset.save_to_disk("data_processed/train_tokenized")
val_dataset.save_to_disk("data_processed/val_tokenized")
test_dataset.save_to_disk("data_processed/test_tokenized")

# También guardar los mappings de labels (son necesarios para el modelo)
import json
with open("data_processed/label_to_id.json", "w") as f:
    json.dump(label_to_id, f)
with open("data_processed/id_to_label.json", "w") as f:
    json.dump(id_to_label, f)

print("Datasets guardados!")

Saving the dataset (1/1 shards): 100%|██████████| 500000/500000 [00:00<00:00, 1441396.07 examples/s]


Datasets guardados!


In [12]:
# Display the first tokenized example from the train_dataset
print(train_dataset[0])

{'text': 'Botella Deportiva Tapa Plata Para Colgar', 'labels': 0, 'input_ids': [4, 2659, 13874, 30932, 12816, 1476, 28765, 30932, 10541, 1830, 3062, 1306, 5], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


#### Create Model and Train

- Define your training hyperparameters in TrainingArguments. At the end of each epoch, the Trainer will evaluate the accuracy and save the training checkpoint.

- Pass the training arguments to Trainer along with the model, dataset and compute_metrics function.

- TODO: Use datacollator().

  - WHY IS TOKENIZER NOT USED in this example? How does it work within the Training?

- Call train() to finetune your model.

In [9]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="beto_model_full_es",
    
    # Learning rate
    learning_rate=2e-5,
    
    # Batch sizes optimizados para RTX 3060 12GB
    per_device_train_batch_size=48,
    per_device_eval_batch_size=32,
    
    # Gradient accumulation (efectivamente batch=32)
    gradient_accumulation_steps=2,
    
    # Epochs
    num_train_epochs=2,
    
    # Regularización
    weight_decay=0.01,
    warmup_ratio=0.1,  # 10% warmup
    
    # Evaluación y guardado
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    
    # OPTIMIZACIONES GPU - RTX 3060
    fp16=True,  # Mixed precision - CRÍTICO para ahorrar VRAM
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    
    # Logging
    logging_dir="./logs",
    logging_steps=500,
    report_to="none",  # Desactivar wandb
    
    # Checkpoints
    save_total_limit=2,
    
    # Misc
    push_to_hub=False,
    seed=42,
)

# Callback para early stopping (opcional pero recomendado)
early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],
)

print("Iniciando entrenamiento...")
print(f"Steps por epoch: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps):,}")

trainer.train()

Iniciando entrenamiento...
Steps por epoch: 93,749


Epoch,Training Loss,Validation Loss,Accuracy
1,0.624400,0.606411,0.871160
2,0.534000,0.539215,0.883228


TrainOutput(global_step=187500, training_loss=0.9419122630208333, metrics={'train_runtime': 71181.2512, 'train_samples_per_second': 252.875, 'train_steps_per_second': 2.634, 'total_flos': 2.8688465266428288e+17, 'train_loss': 0.9419122630208333, 'epoch': 2.0})

In [10]:
trainer.save_model("beto_model_full_es")
print("Modelo guardado en: beto_model_full_es/")

Modelo guardado en: beto_model_full_es/


In [11]:
print("Evaluando en test set...")
results = trainer.evaluate(test_dataset)
print(f"\nResultados en Test:")
for key, value in results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

Evaluando en test set...



Resultados en Test:
  eval_loss: 0.5387
  eval_accuracy: 0.8834
  eval_runtime: 1847.8788
  eval_samples_per_second: 270.5810
  eval_steps_per_second: 8.4560
  epoch: 2.0000


#### Use the model with test dataset

predictions contains the logic (similar to probability) of each category. To get the most likely category np.argmax(predictions.predictions, axis=-1)

In [12]:
# Predicciones en test
predictions = trainer.predict(test_dataset)

predicted_ids = np.argmax(predictions.predictions, axis=-1)
predicted_labels = [id_to_label[i] for i in predicted_ids]

# Mostrar ejemplos
num_examples = 10
print(f"\n{'='*60}")
print(f"Ejemplos de predicciones ({num_examples} muestras):")
print(f"{'='*60}")

for i in range(num_examples):
    true_label = id_to_label[test_dataset['labels'][i]]
    pred_label = predicted_labels[i]
    match = "✓" if true_label == pred_label else "✗"
    
    print(f"\n{match} Texto: {test_dataset['text'][i][:80]}...")
    print(f"   Real: {true_label}")
    print(f"   Pred: {pred_label}")


Ejemplos de predicciones (10 muestras):

✗ Texto: Cubre Motor X-power Ktm 450 Sxf Fc 16 18 Acerbis 23153.061...
   Real: MOTORCYCLE_FAIRINGS
   Pred: MOTORCYCLE_CRASH_BARS

✓ Texto: Lote De 10 Cubiertos  Ac. Inox. Industria Argentina...
   Real: FLATWARE_SETS
   Pred: FLATWARE_SETS

✓ Texto: El Mini Locomotora Mgica Del Tren De La Luz Oro Siguela...
   Real: TOY_TRAINS
   Pred: TOY_TRAINS

✓ Texto: Hidrolavadora Semi Profesional 105 Bar 1200w Lusqtoff Hl120...
   Real: ELECTRIC_PRESSURE_WASHERS
   Pred: ELECTRIC_PRESSURE_WASHERS

✗ Texto: Dmt Dmtw8efhwb Affilacoltelli,unisex - Adultos, ...
   Real: TACTICAL_AND_SPORTING_KNIVES_AND_BLADES
   Pred: LED_STAGE_LIGHTS

✓ Texto: Puerta Corrediza De Embutir 70 X 200 Placa De Cedrillo...
   Real: DOORS
   Pred: DOORS

✗ Texto: Camiseta Lifestyle Hombre Fila Beppe New...
   Real: HATS_AND_CAPS
   Pred: FOOTBALL_SHIRTS

✓ Texto: Kit Espirales Car Traseros Kia Capital Mod.1994 / 1995...
   Real: AUTOMOTIVE_SPRING_SUSPENSIONS
   Pred: AUTOMOTIVE_

#### Metricas adicionales

In [13]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Calcular métricas
true_labels = [id_to_label[test_dataset['labels'][i]] for i in range(len(test_dataset))]

print(f"\n{'='*60}")
print("MÉTRICAS FINALES")
print(f"{'='*60}")
print(f"Accuracy: {accuracy_score(true_labels, predicted_labels):.4f}")
print(f"F1 Macro: {f1_score(true_labels, predicted_labels, average='macro'):.4f}")
print(f"F1 Weighted: {f1_score(true_labels, predicted_labels, average='weighted'):.4f}")

# Classification report (top categorías)
print(f"\n{'='*60}")
print("Classification Report (primeras 20 categorías):")
print(f"{'='*60}")
print(classification_report(true_labels, predicted_labels, zero_division=0))


MÉTRICAS FINALES
Accuracy: 0.8834
F1 Macro: 0.8410
F1 Weighted: 0.8832

Classification Report (primeras 20 categorías):
                                                  precision    recall  f1-score   support

                                      3D_GLASSES       0.80      0.80      0.80        98
                                         3D_PENS       0.70      0.88      0.78        16
                                     3D_PRINTERS       0.88      0.93      0.90       189
                            3D_PRINTER_FILAMENTS       0.96      0.95      0.95       209
                          ABDOMINAL_TONING_BELTS       0.39      0.49      0.43        43
                                     ABS_SENSORS       0.97      0.96      0.97       381
                                AB_ROLLER_WHEELS       0.74      0.56      0.64        50
                                      ACCORDIONS       1.00      0.95      0.97       242
                                ACOUSTIC_GUITARS       0.96      0.9